In [3]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from unidecode import unidecode
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import zscore
from collections import defaultdict
from sklearn.decomposition import PCA
import phik

pds = pandas series  
df = DataFrame

# Preparando o Data Frame

In [4]:
abas  = pd.ExcelFile('WDI_EXCEL/WDIEXCEL.xlsx').sheet_names
abas


['Data', 'Country', 'Series', 'country-series', 'series-time', 'footnote']

In [5]:
metadata = pd.read_excel('WDI_EXCEL/WDIEXCEL.xlsx', sheet_name=abas[2])
metadata.head()

,Series Code,Topic,Indicator Name,Short definition,Long definition,Unit of measure,Periodicity,Base Period,Other notes,Aggregation method,Limitations and exceptions,Notes from original source,General comments,Source,Statistical concept and methodology,Development relevance,Related source links,Other web links,Related indicators,License Type
0,AG.CON.FERT.PT.ZS,Environment: Agricultural production,Fertilizer consumption (% of fertilizer produc...,NaN,Fertilizer consumption measures the quantity o...,NaN,Annual,NaN,The world and regional aggregate series do not...,Weighted average,The FAO has revised the time series for fertil...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Fertilizer consumption measures the quantity o...,"Factors such as the green revolution, has led ...",NaN,NaN,NaN,CC BY-4.0
1,AG.CON.FERT.ZS,Environment: Agricultural production,Fertilizer consumption (kilograms per hectare ...,NaN,Fertilizer consumption measures the quantity o...,NaN,Annual,NaN,The world and regional aggregate series do not...,Weighted average,The FAO has revised the time series for fertil...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Fertilizer consumption measures the quantity o...,"Factors such as the green revolution, has led ...",NaN,NaN,NaN,CC BY-4.0
2,AG.LND.AGRI.K2,Environment: Land use,Agricultural land (sq. km),NaN,Agricultural land refers to the share of land ...,NaN,Annual,NaN,Areas of former states are included in the suc...,Sum,The data are collected by the Food and Agricul...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Agricultural land constitutes only a part of a...,Agricultural land covers more than one-third o...,NaN,NaN,NaN,CC BY-4.0
3,AG.LND.AGRI.ZS,Environment: Land use,Agricultural land (% of land area),NaN,Agricultural land refers to the share of land ...,NaN,Annual,NaN,Areas of former states are included in the suc...,Weighted average,The data are collected by the Food and Agricul...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Agriculture is still a major sector in many ec...,Agricultural land covers more than one-third o...,NaN,NaN,NaN,CC BY-4.0
4,AG.LND.ARBL.HA,Environment: Land use,Arable land (hectares),NaN,Arable land (in hectares) includes land define...,NaN,Annual,NaN,NaN,NaN,The Food and Agriculture Organization (FAO) tr...,NaN,NaN,"Food and Agriculture Organization, electronic ...",Temporary fallow land refers to land left fall...,Agricultural land covers more than one-third o...,NaN,NaN,NaN,CC BY-4.0


In [6]:
metadata.columns = [col.strip().replace("_", " ") for col in metadata.columns]

In [7]:
metadata["Topic"].nunique()

88

In [8]:
lista_metadata = list(metadata["Topic"].unique())

In [9]:
metadata["Indicator Name"].nunique()

1496

In [10]:
dados = pd.read_excel('WDI_EXCEL/WDIEXCEL.xlsx', sheet_name=abas[0])
dados.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.488497,18.001597,18.558234,19.043572,19.586457,20.192064,20.828814,21.372164,22.100884,NaN
1,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,6.811504,7.096003,7.406706,7.666648,8.020952,8.403358,8.718306,9.097176,9.473374,NaN
2,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,38.152090,38.488233,38.779953,39.068462,39.445526,39.818645,40.276374,40.687817,41.211606,NaN
3,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,31.871956,33.922276,38.859598,40.223744,43.035073,44.390861,46.282371,48.127211,48.742043,NaN
4,Africa Eastern and Southern,AFE,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,17.672943,16.527554,24.627753,25.432092,27.061929,29.154282,31.022083,32.809138,33.760782,NaN


In [11]:
dados.columns = [col.strip().replace("_", " ") for col in dados.columns]

In [12]:
print(dados.columns)


Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023'],
      dtype='object')


In [13]:
dados["Indicator Name"].nunique()

1496

In [14]:
dados_long = dados.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="Value"
)

In [15]:
dados_long["Year"] = dados_long["Year"].astype(int)

In [16]:
dados_long.dtypes

Country Name       object
Country Code       object
Indicator Name     object
Indicator Code     object
Year                int64
Value             float64
dtype: object

## Separando por paises e comaçando a olhar o data frame do Brasil

Separação feita desta maneira para uma posssivel extrapolação para demais paises futuramente

In [17]:
paises_unicos = dados_long["Country Name"].unique()

sub_dfs_por_pais = {
    pais: grupo.drop(columns=["Country Code", "Country Name"]).reset_index(drop=True)
    for pais, grupo in dados_long.groupby("Country Name")
}

brasil = sub_dfs_por_pais["Brazil"]
print("\nDados do Brasil:")
brasil.head()


Dados do Brasil:


,Indicator Name,Indicator Code,Year,Value
0,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,1960,NaN
1,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,1960,NaN
2,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,1960,NaN
3,Access to electricity (% of population),EG.ELC.ACCS.ZS,1960,NaN
4,"Access to electricity, rural (% of rural popul...",EG.ELC.ACCS.RU.ZS,1960,NaN


# Analise do Brasil

## Explorando e preparando o data frame para analise

In [368]:
# 1. Filtrar apenas as colunas importantes para a pivotagem
df_brasil = brasil[["Year", "Indicator Name", "Value"]].copy()

# 2. Pivotar para ficar no formato "wide" (uma coluna por indicador)
df_brasil_wide = df_brasil.pivot(
    index="Year",
    columns="Indicator Name",
    values="Value"
)

# 3. Ordenar pelas datas (anos) e exibir as primeiras linhas
df_brasil_wide.sort_index(inplace=True)
df_brasil_wide.head()

Indicator Name,ARI treatment (% of children under 5 taken to a health provider),Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+),"Account ownership at a financial institution or with a mobile-money-service provider, female (% of population ages 15+)","Account ownership at a financial institution or with a mobile-money-service provider, male (% of population ages 15+)",...,Women who believe a husband is justified in beating his wife (any of five reasons) (%),Women who believe a husband is justified in beating his wife when she argues with him (%),Women who believe a husband is justified in beating his wife when she burns the food (%),Women who believe a husband is justified in beating his wife when she goes out without telling him (%),Women who believe a husband is justified in beating his wife when she neglects the children (%),Women who believe a husband is justified in beating his wife when she refuses sex with him (%),Women who were first married by age 15 (% of women ages 20-24),Women who were first married by age 18 (% of women ages 20-24),Women's share of population ages 15+ living with HIV (%),Young people (ages 15-24) newly infected with HIV
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [369]:
pds_br_tps = df_brasil_wide.dtypes
for i ,formato in pds_br_tps.items():
    if formato != 'float64' and formato != 'int64':
        print(i, formato)

### Verificando quantidade de nulos por indicador

In [370]:
# Contar valores nulos por linha
pds_nulos_por_indicador = df_brasil_wide.isna().sum(axis=0)

# Exibir o resultado
print("Quantidade de valores nulos por linha (ano):")
print(pds_nulos_por_indicador.sort_values(ascending=False))

Quantidade de valores nulos por linha (ano):
Indicator Name
Women who believe a husband is justified in beating his wife when she refuses sex with him (%)       64
Vitamin A supplementation coverage rate (% of children ages 6-59 months)                             64
Young people (ages 15-24) newly infected with HIV                                                    64
Educational attainment, at least completed post-secondary, population 25+, male (%) (cumulative)     64
Educational attainment, at least completed post-secondary, population 25+, total (%) (cumulative)    64
                                                                                                     ..
Terms of trade adjustment (constant LCU)                                                              0
Agriculture, forestry, and fishing, value added (constant 2015 US$)                                   0
Agriculture, forestry, and fishing, value added (constant LCU)                                        0
Agri

In [371]:
pds_nulos_por_indicador = pds_nulos_por_indicador.sort_values(ascending=False)
indicadroes_sem_dados = []
for indicador, valor in pds_nulos_por_indicador.items():
    if valor == 64:
        indicadroes_sem_dados.append(indicador)
print(len(indicadroes_sem_dados))

102


In [372]:
new_df_brasil_wide = df_brasil_wide.drop(columns=indicadroes_sem_dados)
new_df_brasil_wide.head()

Indicator Name,ARI treatment (% of children under 5 taken to a health provider),Access to clean fuels and technologies for cooking (% of population),"Access to clean fuels and technologies for cooking, rural (% of rural population)","Access to clean fuels and technologies for cooking, urban (% of urban population)",Access to electricity (% of population),"Access to electricity, rural (% of rural population)","Access to electricity, urban (% of urban population)",Account ownership at a financial institution or with a mobile-money-service provider (% of population ages 15+),"Account ownership at a financial institution or with a mobile-money-service provider, female (% of population ages 15+)","Account ownership at a financial institution or with a mobile-money-service provider, male (% of population ages 15+)",...,"Wage and salaried workers, female (% of female employment) (modeled ILO estimate)","Wage and salaried workers, male (% of male employment) (modeled ILO estimate)","Wage and salaried workers, total (% of total employment) (modeled ILO estimate)",Wanted fertility rate (births per woman),"Water productivity, total (constant 2015 US$ GDP per cubic meter of total freshwater withdrawal)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100),Women who were first married by age 15 (% of women ages 20-24),Women who were first married by age 18 (% of women ages 20-24),Women's share of population ages 15+ living with HIV (%)
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.768224e-13,NaN,NaN,NaN,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.446007e-13,NaN,NaN,NaN,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.748121e-13,NaN,NaN,NaN,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6.508432e-13,NaN,NaN,NaN,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.247164e-12,NaN,NaN,NaN,NaN


In [373]:
pds_nulos_por_indicador = new_df_brasil_wide.isna().sum(axis=0)
pds_nulos_por_indicador = pds_nulos_por_indicador.sort_values(ascending=False)
indicadores_remover = []
for indicador, valor in pds_nulos_por_indicador.items():
    if valor >= (64*0.3):
        indicadores_remover.append(indicador)
print(len(indicadores_remover))

897


In [374]:
new_df_brasil_wide = new_df_brasil_wide.drop(columns=indicadores_remover)
new_df_brasil_wide.head()

Indicator Name,Adjusted net national income (annual % growth),Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (annual % growth),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),...,Travel services (% of commercial service imports),"Travel services (% of service exports, BoP)","Travel services (% of service imports, BoP)","Unemployment, total (% of total labor force) (national estimate)",Urban population,Urban population (% of total population),Urban population growth (annual %),"Use of IMF credit (DOD, current US$)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
Year,,,,,,,,,,,,,,,,,,,,,
1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,33399157.0,46.139,NaN,NaN,1.768224e-13,NaN
1961,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,35155579.0,47.122,5.125267,NaN,2.446007e-13,NaN
1962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,36971452.0,48.099,5.036272,NaN,3.748121e-13,NaN
1963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,38852223.0,49.078,4.961925,NaN,6.508432e-13,NaN
1964,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,40792376.0,50.059,4.872991,NaN,1.247164e-12,NaN


In [375]:
# Seleciona apenas colunas onde o mínimo e o máximo são iguais a 0
colunas_zeradas = new_df_brasil_wide.columns[
    (new_df_brasil_wide.min() == 0) & (new_df_brasil_wide.max() == 0)
]

# Exibe os nomes dessas colunas
print(colunas_zeradas)


Index(['Adjusted savings: net forest depletion (% of GNI)', 'Adjusted savings: net forest depletion (current US$)'], dtype='object', name='Indicator Name')


In [376]:
new_df_brasil_wide = new_df_brasil_wide.drop(columns=colunas_zeradas)

### Verificando quantidade de nulos por ano

In [377]:
# Contar valores nulos por linha
pds_nulos_por_ano = new_df_brasil_wide.isna().sum(axis=1)

# Exibir o resultado
print("Quantidade de valores nulos por linha (ano):")
print(pds_nulos_por_ano)

Quantidade de valores nulos por linha (ano):
Year
1960    260
1961    224
1962    210
1963    208
1964    207
       ... 
2019      7
2020      7
2021     32
2022     66
2023    135
Length: 64, dtype: int64


In [378]:
pds_nulos_por_ano = pds_nulos_por_ano.sort_values(ascending=False)
anos_remover = []
for indicador, valor in pds_nulos_por_ano.items():
    if valor >= (497*0.3):
        anos_remover.append(indicador)
print(len(anos_remover))

10


In [379]:
anos_remover.sort()
anos_remover

[1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969]

In [380]:
new_df_brasil_wide = new_df_brasil_wide.drop(labels=anos_remover, axis=0)
new_df_brasil_wide.head()

Indicator Name,Adjusted net national income (annual % growth),Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (annual % growth),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),...,Travel services (% of commercial service imports),"Travel services (% of service exports, BoP)","Travel services (% of service imports, BoP)","Unemployment, total (% of total labor force) (national estimate)",Urban population,Urban population (% of total population),Urban population growth (annual %),"Use of IMF credit (DOD, current US$)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
Year,,,,,,,,,,,,,,,,,,,,,
1970,NaN,3.291109e+11,3.782767e+10,NaN,3450.680697,396.617704,9.121553,3.806293e+09,3.599282,1.501929e+09,...,NaN,NaN,NaN,NaN,53323573.0,55.909,4.221541,0.0,5.881536e-12,31.250
1971,11.844305,3.680918e+11,4.418975e+10,9.139582,3766.058479,452.118667,9.018367,4.390516e+09,3.599282,1.752280e+09,...,NaN,NaN,NaN,NaN,55607782.0,56.894,4.194465,0.0,7.057843e-12,31.250
1972,12.509426,4.141380e+11,5.269725e+10,9.804705,4135.309393,526.200059,9.028985,5.243614e+09,3.599282,2.090295e+09,...,NaN,NaN,NaN,3.06,57963965.0,57.879,4.149837,0.0,8.369877e-12,31.250
1973,14.291462,4.733244e+11,7.074687e+10,11.557651,4613.254011,689.534076,9.501022,7.446978e+09,3.000000,2.351424e+09,...,NaN,NaN,NaN,2.78,60385804.0,58.855,4.093252,0.0,9.772397e-12,31.250
1974,6.152843,5.024473e+11,9.352216e+10,3.638966,4781.128777,889.927177,9.749445,1.017117e+10,3.599282,3.754972e+09,...,NaN,NaN,NaN,NaN,62870949.0,59.826,4.033015,0.0,1.262268e-11,28.125


In [381]:
colunas_preservadas = new_df_brasil_wide[['Population growth (annual %)', 'Population, total']].copy()

### Verificando indicadores redundantes

In [382]:
new_df_brasil_wide.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54 entries, 1970 to 2023
Columns: 495 entries, Adjusted net national income (annual % growth) to Women Business and the Law Index Score (scale 1-100)
dtypes: float64(495)
memory usage: 209.2 KB


In [383]:
new_df_brasil_wide.describe()

Indicator Name,Adjusted net national income (annual % growth),Adjusted net national income (constant 2015 US$),Adjusted net national income (current US$),Adjusted net national income per capita (annual % growth),Adjusted net national income per capita (constant 2015 US$),Adjusted net national income per capita (current US$),Adjusted savings: consumption of fixed capital (% of GNI),Adjusted savings: consumption of fixed capital (current US$),Adjusted savings: education expenditure (% of GNI),Adjusted savings: education expenditure (current US$),...,Travel services (% of commercial service imports),"Travel services (% of service exports, BoP)","Travel services (% of service imports, BoP)","Unemployment, total (% of total labor force) (national estimate)",Urban population,Urban population (% of total population),Urban population growth (annual %),"Use of IMF credit (DOD, current US$)",Wholesale price index (2010 = 100),Women Business and the Law Index Score (scale 1-100)
count,51.000000,5.200000e+01,5.200000e+01,51.000000,52.000000,52.000000,52.000000,5.200000e+01,52.000000,5.200000e+01,...,49.000000,49.000000,49.000000,45.000000,5.400000e+01,54.000000,54.000000,5.400000e+01,5.200000e+01,54.000000
mean,3.010929,9.162268e+11,6.681062e+11,1.425942,5590.578090,3682.081997,14.456639,1.461168e+11,4.417439,4.287812e+10,...,19.294171,16.320850,18.161931,7.394556,1.256833e+08,75.951296,2.385401,4.433198e+09,5.155756e+01,57.060185
std,6.368678,3.304046e+11,5.915988e+11,6.130710,916.290481,2801.135806,3.945016,1.645337e+11,0.982101,4.571928e+10,...,7.152482,9.000527,7.024541,3.469044,4.176731e+07,9.722235,1.177507,6.723700e+09,6.612087e+01,22.270061
min,-14.620223,3.291109e+11,3.782767e+10,-16.131683,3450.680697,396.617704,9.018367,3.806293e+09,2.800000,1.501929e+09,...,6.648368,2.262181,5.898268,1.920000,5.332357e+07,55.909000,0.632380,0.000000e+00,5.881536e-12,28.125000
25%,-0.211241,6.445987e+11,1.925976e+11,-1.653632,4987.951719,1490.704292,11.792705,2.432008e+10,3.599282,7.718184e+09,...,14.728987,7.253886,13.296869,4.040000,8.921814e+07,68.359000,1.201713,8.609384e+07,1.327366e-09,33.125000
50%,3.097826,9.307267e+11,4.434869e+11,1.276949,5542.382954,2537.679504,13.756643,7.698704e+10,4.264303,2.063184e+10,...,19.942503,17.324395,18.475186,7.578000,1.301583e+08,78.675000,2.422724,2.533385e+09,2.191907e+01,50.625000
75%,5.882612,1.271165e+12,1.160585e+12,4.064897,6433.397369,5848.704142,15.720799,2.527057e+11,5.217500,8.456528e+10,...,24.342780,22.045219,22.862175,10.150000,1.628768e+08,84.262250,3.513118,4.443875e+09,9.463679e+01,81.875000
max,22.522958,1.435186e+12,2.003637e+12,20.438679,7230.947942,10260.080844,23.062757,5.187513e+11,6.281311,1.399386e+11,...,35.950176,39.080460,34.497130,13.697000,1.853562e+08,87.788000,4.221541,2.885033e+10,2.752110e+02,85.000000


In [384]:
# Calcular a matriz de correlação para new_df_brasil_wide
corr_matrix = new_df_brasil_wide.corr().abs()

# Remover a diagonal principal (correlação de uma variável com ela mesma)
np.fill_diagonal(corr_matrix.values, 0)

# Para cada variável, encontrar a maior correlação com outra variável
max_corr = corr_matrix.max()

# Ordenar as variáveis por valor máximo de correlação (em ordem decrescente)
high_corr_vars = max_corr.sort_values(ascending=False)

# Mostrar os top 20 indicadores com correlações mais altas
print("Top 20 indicadores com correlações mais altas:")
print(high_corr_vars.head(20))


Top 20 indicadores com correlações mais altas:
Indicator Name
Agriculture, forestry, and fishing, value added (constant 2015 US$)                                         1.0
Agriculture, forestry, and fishing, value added (constant LCU)                                              1.0
Gross capital formation (constant LCU)                                                                      1.0
Gross capital formation (constant 2015 US$)                                                                 1.0
GDP (constant LCU)                                                                                          1.0
Industry (including construction), value added (constant 2015 US$)                                          1.0
GDP (constant 2015 US$)                                                                                     1.0
Industry (including construction), value added (constant LCU)                                               1.0
Services, value added (constant LCU)      

In [385]:
alta_correlacao = []
for col in new_df_brasil_wide.columns:
    # Encontra os pares com correlação > 0.9 para esta coluna
    correlacionados = corr_matrix[col][corr_matrix[col] > 0.9].index.tolist()
    if correlacionados:  # Se houver alguma correlação alta
        alta_correlacao.append(col)

print(f"Número de variáveis com alta correlação (>0.9): {len(alta_correlacao)}")

Número de variáveis com alta correlação (>0.9): 387


In [386]:
new_df_brasil_wide.drop(columns=alta_correlacao, inplace=True)

In [387]:
new_df_brasil_wide.head()

Indicator Name,Adjusted savings: mineral depletion (current US$),Adjusted savings: net national savings (% of GNI),Adjusted savings: net national savings (current US$),Agricultural raw materials exports (% of merchandise exports),Agricultural raw materials imports (% of merchandise imports),"Agriculture, forestry, and fishing, value added (% of GDP)","Agriculture, forestry, and fishing, value added (annual % growth)",Arms exports (SIPRI trend indicator values),Arms imports (SIPRI trend indicator values),Average precipitation in depth (mm per year),...,"Short-term debt (% of exports of goods, services and primary income)",Short-term debt (% of total external debt),Short-term debt (% of total reserves),Surface area (sq. km),Terms of trade adjustment (constant LCU),Total debt service (% of GNI),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,"Unemployment, total (% of total labor force) (national estimate)","Use of IMF credit (DOD, current US$)"
Year,,,,,,,,,,,,,,,,,,,,,
1970,7.468548e+07,NaN,NaN,11.895098,1.851044,0.0,5.600000,NaN,219000000.0,1782.0,...,NaN,11.9318,60.006561,8515770.0,1.385222e+10,1.838590,NaN,NaN,NaN,0.0
1971,6.975861e+07,NaN,NaN,10.494023,2.178320,0.0,10.154178,0.0,179000000.0,1782.0,...,NaN,11.9465,52.627887,8515770.0,1.067689e+10,1.805230,NaN,NaN,NaN,0.0
1972,9.610341e+07,NaN,NaN,9.322425,1.971959,0.0,3.967431,NaN,518000000.0,1761.0,...,NaN,11.9275,33.541404,8515770.0,1.393160e+10,2.072381,NaN,NaN,3.06,0.0
1973,1.175452e+08,NaN,NaN,8.521143,2.073260,0.0,0.075501,NaN,598000000.0,1782.0,...,NaN,11.9544,27.670790,8515770.0,2.787188e+10,2.453171,NaN,NaN,2.78,0.0
1974,2.272234e+08,NaN,NaN,5.991149,2.014877,0.0,1.299594,32000000.0,256000000.0,1782.0,...,NaN,12.0278,49.458973,8515770.0,8.881354e+09,3.282392,NaN,NaN,NaN,0.0


In [388]:
new_df_brasil_wide.describe()

Indicator Name,Adjusted savings: mineral depletion (current US$),Adjusted savings: net national savings (% of GNI),Adjusted savings: net national savings (current US$),Agricultural raw materials exports (% of merchandise exports),Agricultural raw materials imports (% of merchandise imports),"Agriculture, forestry, and fishing, value added (% of GDP)","Agriculture, forestry, and fishing, value added (annual % growth)",Arms exports (SIPRI trend indicator values),Arms imports (SIPRI trend indicator values),Average precipitation in depth (mm per year),...,"Short-term debt (% of exports of goods, services and primary income)",Short-term debt (% of total external debt),Short-term debt (% of total reserves),Surface area (sq. km),Terms of trade adjustment (constant LCU),Total debt service (% of GNI),"Total debt service (% of exports of goods, services and primary income)",Total reserves in months of imports,"Unemployment, total (% of total labor force) (national estimate)","Use of IMF credit (DOD, current US$)"
count,5.200000e+01,47.000000,4.700000e+01,53.000000,53.000000,54.000000,54.000000,4.700000e+01,5.400000e+01,52.000000,...,49.000000,54.000000,54.000000,5.300000e+01,5.400000e+01,54.000000,49.000000,49.000000,45.000000,5.400000e+01
mean,3.322432e+09,2.875598,-7.379530e+09,4.403473,1.727535,4.918723,3.695192,8.495745e+07,2.682222e+08,1773.115385,...,37.305239,13.483259,80.082799,8.515669e+06,-2.110421e+10,5.180753,45.612839,7.298229,7.394556,4.433198e+09
std,4.985346e+09,6.222749,6.161769e+10,1.852834,0.736431,3.222175,4.846776,6.673274e+07,1.883150e+08,10.476217,...,18.141000,3.761722,84.754875,7.351852e+02,2.912410e+10,2.757800,20.588254,4.019399,3.469044,6.723700e+09
min,6.975861e+07,-9.350084,-1.643374e+11,2.273232,0.796975,0.000000,-8.020000,0.000000e+00,5.700000e+07,1761.000000,...,11.255509,6.926600,8.731629,8.510418e+06,-7.986852e+10,1.805230,15.731413,1.206051,1.920000,0.000000e+00
25%,5.950463e+08,-0.543720,-4.701047e+09,3.514469,1.158042,4.233548,0.819771,3.250000e+07,1.350000e+08,1761.000000,...,22.710116,10.692775,21.868827,8.515770e+06,-4.392339e+10,2.909652,29.916404,4.143631,4.040000,8.609384e+07
50%,8.172859e+08,3.834196,1.367722e+10,3.882940,1.495779,4.717546,3.280666,5.400000e+07,2.210000e+08,1782.000000,...,33.263961,12.321100,59.862100,8.515770e+06,-1.733534e+10,4.502670,43.694779,6.606328,7.578000,2.533385e+09
75%,4.673545e+09,6.671306,2.050667e+10,4.735486,2.073260,6.479418,6.336670,1.250000e+08,3.457500e+08,1782.000000,...,49.714194,15.355925,91.048272,8.515770e+06,2.018321e+09,6.984783,54.915436,10.526578,10.150000,4.443875e+09
max,2.322162e+10,21.878337,8.188657e+10,11.895098,4.318985,11.857332,15.120991,2.750000e+08,8.720000e+08,1782.000000,...,74.495791,21.685000,437.374535,8.515770e+06,3.638676e+10,11.733190,116.563894,16.603420,13.697000,2.885033e+10


In [389]:
# Remove colunas com apenas 1 valor distinto (ex: só zeros, ou valor constante)
new_df_brasil_wide = new_df_brasil_wide.loc[:, new_df_brasil_wide.nunique() > 1]


In [390]:
# calcula a matriz phik
n_df_for_phik = new_df_brasil_wide.fillna(0)
phik_matrix = n_df_for_phik.phik_matrix()

interval columns not set, guessing: ['Adjusted savings: mineral depletion (current US$)', 'Adjusted savings: net national savings (% of GNI)', 'Adjusted savings: net national savings (current US$)', 'Agricultural raw materials exports (% of merchandise exports)', 'Agricultural raw materials imports (% of merchandise imports)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Agriculture, forestry, and fishing, value added (annual % growth)', 'Arms exports (SIPRI trend indicator values)', 'Arms imports (SIPRI trend indicator values)', 'Average precipitation in depth (mm per year)', 'Broad money (% of GDP)', 'Broad money to total reserves ratio', 'Capture fisheries production (metric tons)', 'Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)', 'Changes in inventories (current LCU)', 'Changes in inventories (current US$)', 'Claims on central government (annual growth as % of broad money)', 'Claims on central government, etc. (% GDP)', 'Current account balance (%

In [391]:
# Calcula a matriz de correlação Phi-K e converte para valores absolutos
corr_matrix_phik = new_df_brasil_wide.phik_matrix().abs()

# Remove a diagonal principal (correlação de uma variável com ela mesma)
np.fill_diagonal(corr_matrix_phik.values, 0)

# Lista para armazenar variáveis com alta correlação
alta_correlacao_phik = []
# Dicionário para armazenar quais variáveis estão correlacionadas com cada coluna
pares_correlacionados = {}

for col in new_df_brasil_wide.columns:
    if col in corr_matrix_phik.columns:
    # Encontra os pares com correlação > 0.9 para esta coluna
        correlacionados = corr_matrix_phik[col][corr_matrix_phik[col] > 0.9].index.tolist()
        if correlacionados:  # Se houver alguma correlação alta
            alta_correlacao_phik.append(col)
            pares_correlacionados[col] = correlacionados

print(f"Número de variáveis com alta correlação (>0.9): {len(alta_correlacao)}")

interval columns not set, guessing: ['Adjusted savings: mineral depletion (current US$)', 'Adjusted savings: net national savings (% of GNI)', 'Adjusted savings: net national savings (current US$)', 'Agricultural raw materials exports (% of merchandise exports)', 'Agricultural raw materials imports (% of merchandise imports)', 'Agriculture, forestry, and fishing, value added (% of GDP)', 'Agriculture, forestry, and fishing, value added (annual % growth)', 'Arms exports (SIPRI trend indicator values)', 'Arms imports (SIPRI trend indicator values)', 'Average precipitation in depth (mm per year)', 'Broad money (% of GDP)', 'Broad money to total reserves ratio', 'Capture fisheries production (metric tons)', 'Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP)', 'Changes in inventories (current LCU)', 'Changes in inventories (current US$)', 'Claims on central government (annual growth as % of broad money)', 'Claims on central government, etc. (% GDP)', 'Current account balance (%

In [392]:
new_df_brasil_wide.drop(columns=alta_correlacao_phik, inplace=True)

In [393]:
new_df_brasil_wide.head()

Indicator Name,Adjusted savings: net national savings (% of GNI),Adjusted savings: net national savings (current US$),"Agriculture, forestry, and fishing, value added (% of GDP)","Agriculture, forestry, and fishing, value added (annual % growth)",Arms exports (SIPRI trend indicator values),Arms imports (SIPRI trend indicator values),Average precipitation in depth (mm per year),Broad money (% of GDP),Capture fisheries production (metric tons),Carbon intensity of GDP (kg CO2e per constant 2015 US$ of GDP),...,Rural population growth (annual %),"Services, value added (% of GDP)",Sex ratio at birth (male births per female births),"Short-term debt (% of exports of goods, services and primary income)",Short-term debt (% of total external debt),Terms of trade adjustment (constant LCU),Total debt service (% of GNI),"Total debt service (% of exports of goods, services and primary income)","Unemployment, total (% of total labor force) (national estimate)","Use of IMF credit (DOD, current US$)"
Year,,,,,,,,,,,,,,,,,,,,,
1970,NaN,NaN,0.0,5.600000,NaN,219000000.0,1782.0,19.408147,573021.0,0.311786,...,0.299246,0.0,1.046,NaN,11.9318,1.385222e+10,1.838590,NaN,NaN,0.0
1971,NaN,NaN,0.0,10.154178,0.0,179000000.0,1782.0,20.131933,613442.0,0.279089,...,0.188666,0.0,1.045,NaN,11.9465,1.067689e+10,1.805230,NaN,NaN,0.0
1972,NaN,NaN,0.0,3.967431,NaN,518000000.0,1761.0,18.754646,620117.0,0.271842,...,0.121785,0.0,1.045,NaN,11.9275,1.393160e+10,2.072381,NaN,3.06,0.0
1973,NaN,NaN,0.0,0.075501,NaN,598000000.0,1782.0,18.169945,759919.0,0.277423,...,0.076637,0.0,1.045,NaN,11.9544,2.787188e+10,2.453171,NaN,2.78,0.0
1974,NaN,NaN,0.0,1.299594,32000000.0,256000000.0,1782.0,16.909661,715030.0,0.276981,...,0.008418,0.0,1.045,NaN,12.0278,8.881354e+09,3.282392,NaN,NaN,0.0


In [394]:
new_df_brasil_wide = pd.concat([new_df_brasil_wide, colunas_preservadas], axis=1)

## Condensando as variaveis

### Topico com subtopico

In [395]:
# Filtra indicadores da metadata que estão no df_brasil_wide
metadata_filtered = metadata[metadata["Indicator Name"].isin(new_df_brasil_wide.columns)]

# Agrupa por tópico
agrupamentos_por_topico = (
    metadata_filtered.groupby("Topic")["Indicator Name"]
    .apply(list)
    .to_dict()
)


In [396]:
x = []
for topico in agrupamentos_por_topico.keys():
    z = []
    for item in agrupamentos_por_topico[topico]:
      z.append(item)
    x.append(len(z))
print(x)

[4, 1, 2, 1, 7, 4, 2, 6, 1, 3, 5, 12, 1, 1, 2, 1, 1, 1, 1, 2, 5, 4, 3, 1, 1]


In [397]:
# 3. Cria um dicionário de DataFrames por tópico
dfs_por_topico = {}

for topico, indicadores in agrupamentos_por_topico.items():
    # Garante que os indicadores existem no DataFrame
    indicadores_validos = [ind for ind in indicadores if ind in new_df_brasil_wide.columns]
    
    if indicadores_validos:
        df_topico = new_df_brasil_wide[indicadores_validos].copy()
        dfs_por_topico[topico] = df_topico


In [398]:
diagnostico_topico = []

for topico, df in dfs_por_topico.items():
    df_clean = df.dropna(axis=1, how='all').dropna()  # remove colunas completamente nulas e linhas com NA

    if df_clean.empty:
        continue

    # Padroniza os dados
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_clean), columns=df_clean.columns)

    # Contagem total de outliers (z-score > 3)
    outliers_mask = df_scaled.abs() > 3
    n_outliers_total = outliers_mask.sum().sum()

    # Quantas variáveis têm ao menos 1 outlier
    n_vars_outlier = (outliers_mask.sum() > 0).sum()
    pct_vars_outlier = round((n_vars_outlier / len(df_clean)), 2)

    # Correlações
    try:
        corr_matrix = df_scaled.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        n_corr_alta = (upper_tri > 0.9).sum().sum()
        n_vars = df_scaled.shape[1]
        if n_vars <= 1:
            per_cent_corr = 0.0
        else:
            total_possible = n_vars * (n_vars - 1) / 2
            per_cent_corr = round((n_corr_alta / total_possible) * 100, 2)
    except:
        per_cent_corr = None

    diagnostico_topico.append({
        'Topico': topico,
        'N_Variáveis': df.shape[1],
        'N_Observações': df.shape[0],
        '% Missing': round(df.isnull().mean().mean() * 100, 2),
        '% correlações > 0.9': per_cent_corr,
        'N_Outliers (z > 3)': int(n_outliers_total),
        '% Variáveis com Outliers': pct_vars_outlier
    })

df_diag = pd.DataFrame(diagnostico_topico).sort_values('Topico', ascending=True).reset_index(drop=True)
df_diag

,Topico,N_Variáveis,N_Observações,% Missing,% correlações > 0.9,N_Outliers (z > 3),% Variáveis com Outliers
0,Economic Policy & Debt: Balance of payments: C...,4,54,6.94,0.0,4,0.06
1,Economic Policy & Debt: Balance of payments: C...,1,54,9.26,0.0,0,0.00
2,Economic Policy & Debt: Balance of payments: C...,2,54,9.26,0.0,0,0.00
3,Economic Policy & Debt: External debt: Debt ou...,1,54,0.00,0.0,2,0.02
4,Economic Policy & Debt: External debt: Debt ra...,7,54,2.65,0.0,2,0.04
5,Economic Policy & Debt: External debt: Net flows,4,54,1.85,0.0,5,0.08
6,Economic Policy & Debt: National accounts: Adj...,2,54,12.96,0.0,1,0.02
7,Economic Policy & Debt: National accounts: Gro...,6,54,0.62,0.0,3,0.06
8,Economic Policy & Debt: National accounts: Loc...,1,54,0.00,0.0,0,0.00
9,Economic Policy & Debt: National accounts: Loc...,3,54,0.00,0.0,3,0.06


### Topicos macros

In [399]:
dfs_por_macrotema = defaultdict(list)

for topico, df in dfs_por_topico.items():
    macrotema = topico.split(':')[0].strip()
    dfs_por_macrotema[macrotema].append(df)


In [400]:
dfs_macrotema_unificado = {}

for macrotema, lista_dfs in dfs_por_macrotema.items():
    df_merged = pd.concat(lista_dfs, axis=1)
    # Remove colunas duplicadas (caso haja) e mantém ordem
    df_merged = df_merged.loc[:, ~df_merged.columns.duplicated()]
    dfs_macrotema_unificado[macrotema] = df_merged

In [401]:
diagnostico_macrotema = []

for tema, df in dfs_macrotema_unificado.items():
    df_clean = df.dropna(axis=1, how='all').dropna()  # remove colunas completamente nulas e linhas com NA

    if df_clean.empty:
        continue

    # Padroniza os dados
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df_clean), columns=df_clean.columns)

    # Contagem total de outliers (z-score > 3)
    outliers_mask = df_scaled.abs() > 3
    n_outliers_total = outliers_mask.sum().sum()

    # Quantas variáveis têm ao menos 1 outlier
    n_vars_outlier = (outliers_mask.sum() > 0).sum()
    pct_vars_outlier = round((n_vars_outlier / len(df_clean)), 2)

    # Correlações
    try:
        corr_matrix = df_scaled.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        n_corr_alta = (upper_tri > 0.9).sum().sum()
        n_vars = df_scaled.shape[1]
        if n_vars <= 1:
            per_cent__corr = 0.0  # Ou np.nan, dependendo do que quiser representar
        else:
            total_possible = n_vars * (n_vars - 1) / 2
            per_cent__corr = round((n_corr_alta / total_possible) * 100, 2)

    except:
        per_cent__corr = None

    diagnostico_macrotema.append({
        'Macrotema': tema,
        'N_Variáveis': df.shape[1],
        'N_Observações': df.shape[0],
        '% Missing': round(df.isnull().mean().mean() * 100, 2),
        '% correlações > 0.9': per_cent__corr,
        'N_Outliers (z > 3)': int(n_outliers_total),
        '% Variáveis com Outliers': pct_vars_outlier
    })

df_diag_macro = pd.DataFrame(diagnostico_macrotema).sort_values('N_Variáveis', ascending=False)
df_diag_macro.reset_index(drop=True, inplace=True)
df_diag_macro


,Macrotema,N_Variáveis,N_Observações,% Missing,% correlações > 0.9,N_Outliers (z > 3),% Variáveis com Outliers
0,Economic Policy & Debt,48,54,3.40,0.00,23,0.55
1,Private Sector & Trade,9,54,4.32,0.00,5,0.10
2,Environment,5,54,2.22,0.00,0,0.00
3,Public Sector,3,54,4.32,0.00,1,0.02
4,Health,3,54,0.62,33.33,0,0.00
5,Financial Sector,2,54,1.85,0.00,0,0.00
6,Social Protection & Labor,2,54,8.33,0.00,0,0.00
